In [3]:
%pip install python-dotenv langchain-upstage


Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install langchain langchain-community langchain-text-splitters langchain-pinecone docx2txt

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install python-docx

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------- ----------------------------- 1.0/4.0 MB 6.8 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.0 MB 5.3 MB/s eta 0:00:01
   --------------------------------- ------ 3.4/4.0 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 5.7 MB/s  0:00:00

   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- -------------

In [ ]:
# 접구선통기술기준 파이콘에서 삭제후 저장, 메타데이터(법률/몇조/몇항)추가해서 저장

import os
import re
import docx2txt
from typing import List, Tuple

from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_upstage import UpstageEmbeddings
from langchain_pinecone import PineconeVectorStore

import re
import hashlib
from typing import List

load_dotenv()

# -----------------------------
# 0) 환경변수
# -----------------------------
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("GROUNDLINE_INDEX")   # 또는 재저장할 인덱스명
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "default")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY가 없습니다.")
if not INDEX_NAME:
    raise ValueError("INDEX_NAME(GROUNDLINE_INDEX)가 없습니다.")

DOCX_PATH = "./접구선통기술기준_20251128.docx"
DOC_ID = "접구선통기술기준"
REGULATION_NAME = "접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준"

# -----------------------------
# 1) Pinecone 연결
# -----------------------------
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)

embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE,
)

# -----------------------------
# 2) 원문 추출
# -----------------------------
def load_docx_text(path: str) -> str:
    text = docx2txt.process(path)
    if not text or not text.strip():
        raise ValueError(f"문서에서 텍스트를 추출하지 못했습니다: {path}")
    return text

# -----------------------------
# 3) 조문/항 파싱
#    - 제n조(제목) 단위 분리
#    - 각 조 안에서 제n항 단위 분리
# -----------------------------
# ARTICLE_PATTERN = re.compile(r"(제\s*\d+\s*조(?:의\s*\d+)?)(\s*\([^)]*\))?")
ARTICLE_PATTERN = re.compile(
    r"^\s*(제\s*\d+\s*조(?:의\s*\d+)?)(\s*\([^)]*\))?",
    re.MULTILINE
)
CLAUSE_PATTERN = re.compile(r"(제\s*\d+\s*항)")

def normalize_space(text: str) -> str:
    return re.sub(r"[ \t]+", " ", text).strip()

def split_articles(full_text: str) -> List[Tuple[str, str]]:
    """
    반환: [(article, article_text), ...]
    """
    matches = list(ARTICLE_PATTERN.finditer(full_text))
    results = []

    if not matches:
        return results

    for i, m in enumerate(matches):
        article = normalize_space(m.group(1))
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        article_text = full_text[start:end].strip()
        results.append((article, article_text))

    return results

def split_clauses(article_text: str) -> List[Tuple[str, str]]:
    """
    반환: [(clause, clause_text), ...]
    항이 없으면 빈 리스트 반환
    """
    matches = list(CLAUSE_PATTERN.finditer(article_text))
    results = []

    if not matches:
        return results

    for i, m in enumerate(matches):
        clause = normalize_space(m.group(1))
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(article_text)
        clause_text = article_text[start:end].strip()
        results.append((clause, clause_text))

    return results

def parse_law_to_documents(
    full_text: str,
    regulation_name: str,
    source: str,
    doc_id: str,
) -> List[Document]:
    docs: List[Document] = []

    articles = split_articles(full_text)

    if not articles:
        raise ValueError("조문(제n조) 패턴을 찾지 못했습니다. 문서 형식을 확인하세요.")

    for article, article_text in articles:
        # clauses = split_clauses(article_text)
        clauses = []


        # 항이 있는 경우: 항 단위 저장
        if clauses:
            for clause, clause_text in clauses:
                docs.append(
                    Document(
                        page_content=normalize_space(clause_text),
                        metadata={
                            "doc_id": doc_id,
                            "regulation_name": regulation_name,
                            "article": article,
                            "clause": clause,
                            "source": source,
                        },
                    )
                )
        else:
            # 항이 없으면 조 단위 저장
            docs.append(
                Document(
                    page_content=normalize_space(article_text),
                    metadata={
                        "doc_id": doc_id,
                        "regulation_name": regulation_name,
                        "article": article,
                        "clause": "",
                        "source": source,
                    },
                )
            )

    return docs

# -----------------------------
# 4) ID 생성
# -----------------------------
def ascii_safe(text: str) -> str:
    """
    Pinecone ID용 ASCII 안전 문자열 생성
    한글/특수문자가 있으면 제거하고, 비어 있으면 해시 사용
    """
    if not text:
        return "empty"

    # 영문, 숫자, -, _, :, . 만 남김
    cleaned = re.sub(r"[^A-Za-z0-9_\-:.]", "", text)

    if cleaned:
        return cleaned

    # 전부 제거되면 해시로 대체
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:12]


def build_ids(docs: List[Document]) -> List[str]:
    ids = []

    for i, doc in enumerate(docs, start=1):
        doc_id = ascii_safe(doc.metadata.get("doc_id", "doc"))
        article = ascii_safe(doc.metadata.get("article", ""))
        clause = ascii_safe(doc.metadata.get("clause", ""))

        vector_id = f"{doc_id}::{article}::{clause}::{i:05d}"
        ids.append(vector_id)

    return ids


# -----------------------------
# 5) 기존 데이터 삭제
#    doc_id 기준 삭제
# -----------------------------
def delete_existing_doc(doc_id: str):
    print(f"기존 데이터 삭제 중... doc_id={doc_id}")
    index.delete(
        namespace=NAMESPACE,
        filter={"doc_id": {"$eq": doc_id}},
    )
    print("기존 데이터 삭제 완료")

# -----------------------------
# 6) 새 문서 업로드
# -----------------------------
def reindex_law_docx():
    # 1. 문서 텍스트 추출
    full_text = load_docx_text(DOCX_PATH)
    
    # 2. 기존 데이터 삭제
    delete_existing_doc(DOC_ID)

    # 3. 조문 재파싱
    docs = parse_law_to_documents(
        full_text=full_text,
        regulation_name=REGULATION_NAME,
        source=os.path.basename(DOCX_PATH),
        doc_id=DOC_ID,
    )

    # 4. ID 생성
    ids = build_ids(docs)

    # 5. 재저장
    vectorstore.add_documents(documents=docs, ids=ids)

    print(f"재저장 완료: {len(docs)}건")
    print("샘플 metadata:")
    for d in docs[:3]:
        print(d.metadata)

    return docs, ids

# -----------------------------
# 7) 실행
# -----------------------------
docs, ids = reindex_law_docx()

# -----------------------------
# 8) docs 내용을 output.docx로 저장 (한글 폰트 지정)
# -----------------------------
from docx import Document as DocxDocument
from docx.shared import Pt
from docx.oxml.ns import qn

def save_docs_to_docx(docs: List[Document], output_path: str = "output.docx"):
    docx = DocxDocument()
    docx.add_heading("법령 파싱 결과", level=0)

    for d in docs:
        regulation = d.metadata.get("regulation_name", "")
        article = d.metadata.get("article", "")
        clause = d.metadata.get("clause", "")

        title = f"{regulation} {article} {clause}".strip()

        # 제목
        heading = docx.add_heading(level=2)
        run = heading.add_run(title)
        run.font.name = "맑은 고딕"
        run._element.rPr.rFonts.set(qn('w:eastAsia'), '맑은 고딕')

        # 본문
        p = docx.add_paragraph()
        run = p.add_run(d.page_content)
        run.font.name = "맑은 고딕"
        run._element.rPr.rFonts.set(qn('w:eastAsia'), '맑은 고딕')
        run.font.size = Pt(11)

        docx.add_paragraph("")

    docx.save(output_path)
    print(f"DOCX 저장 완료: {output_path}")


# docs 내용을 docx로 저장
# save_docs_to_docx(docs)

c:\Users\호두마루\Desktop\project\communication-law\venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
c:\Users\호두마루\Desktop\project\communication-law\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


재저장 완료: 78건
샘플 metadata:
{'doc_id': '접구선통기술기준', 'regulation_name': '접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준', 'article': '제1조', 'clause': '', 'source': '접구선통기술기준_20251128.docx'}
{'doc_id': '접구선통기술기준', 'regulation_name': '접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준', 'article': '제2조', 'clause': '', 'source': '접구선통기술기준_20251128.docx'}
{'doc_id': '접구선통기술기준', 'regulation_name': '접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준', 'article': '제3조', 'clause': '', 'source': '접구선통기술기준_20251128.docx'}
DOCX 저장 완료: output.docx


In [19]:
# 방송통신설비기술기준 파이콘에서 삭제후 저장, 메타데이터(법률/몇조/몇항)추가해서 저장

import os
import re
import docx2txt
from typing import List, Tuple

from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_upstage import UpstageEmbeddings
from langchain_pinecone import PineconeVectorStore

import re
import hashlib
from typing import List

load_dotenv()

# -----------------------------
# 0) 환경변수
# -----------------------------
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("BROADCOM_INDEX")   # 또는 재저장할 인덱스명
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "default")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY가 없습니다.")
if not INDEX_NAME:
    raise ValueError("INDEX_NAME(BROADCOM_INDEX)가 없습니다.")

DOCX_PATH = "./방송통신설비기술기준_20240719.docx"
DOC_ID = "방송통신설비기술기준"
REGULATION_NAME = "방송통신설비의 기술기준에 관한 규정"

# -----------------------------
# 1) Pinecone 연결
# -----------------------------
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)

embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE,
)

# -----------------------------
# 2) 원문 추출
# -----------------------------
def load_docx_text(path: str) -> str:
    text = docx2txt.process(path)
    if not text or not text.strip():
        raise ValueError(f"문서에서 텍스트를 추출하지 못했습니다: {path}")
    return text

# -----------------------------
# 3) 조문/항 파싱
#    - 제n조(제목) 단위 분리
#    - 각 조 안에서 제n항 단위 분리
# -----------------------------
ARTICLE_PATTERN = re.compile(r"(제\s*\d+\s*조(?:의\s*\d+)?)(\s*\([^)]*\))?")
CLAUSE_PATTERN = re.compile(r"(제\s*\d+\s*항)")

def normalize_space(text: str) -> str:
    return re.sub(r"[ \t]+", " ", text).strip()

def split_articles(full_text: str) -> List[Tuple[str, str]]:
    """
    반환: [(article, article_text), ...]
    """
    matches = list(ARTICLE_PATTERN.finditer(full_text))
    results = []

    if not matches:
        return results

    for i, m in enumerate(matches):
        article = normalize_space(m.group(1))
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        article_text = full_text[start:end].strip()
        results.append((article, article_text))

    return results

def split_clauses(article_text: str) -> List[Tuple[str, str]]:
    """
    반환: [(clause, clause_text), ...]
    항이 없으면 빈 리스트 반환
    """
    matches = list(CLAUSE_PATTERN.finditer(article_text))
    results = []

    if not matches:
        return results

    for i, m in enumerate(matches):
        clause = normalize_space(m.group(1))
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(article_text)
        clause_text = article_text[start:end].strip()
        results.append((clause, clause_text))

    return results

def parse_law_to_documents(
    full_text: str,
    regulation_name: str,
    source: str,
    doc_id: str,
) -> List[Document]:
    docs: List[Document] = []

    articles = split_articles(full_text)

    if not articles:
        raise ValueError("조문(제n조) 패턴을 찾지 못했습니다. 문서 형식을 확인하세요.")

    for article, article_text in articles:
        clauses = split_clauses(article_text)

        # 항이 있는 경우: 항 단위 저장
        if clauses:
            for clause, clause_text in clauses:
                docs.append(
                    Document(
                        page_content=normalize_space(clause_text),
                        metadata={
                            "doc_id": doc_id,
                            "regulation_name": regulation_name,
                            "article": article,
                            "clause": clause,
                            "source": source,
                        },
                    )
                )
        else:
            # 항이 없으면 조 단위 저장
            docs.append(
                Document(
                    page_content=normalize_space(article_text),
                    metadata={
                        "doc_id": doc_id,
                        "regulation_name": regulation_name,
                        "article": article,
                        "clause": "",
                        "source": source,
                    },
                )
            )

    return docs

# -----------------------------
# 4) ID 생성
# -----------------------------
def ascii_safe(text: str) -> str:
    """
    Pinecone ID용 ASCII 안전 문자열 생성
    한글/특수문자가 있으면 제거하고, 비어 있으면 해시 사용
    """
    if not text:
        return "empty"

    # 영문, 숫자, -, _, :, . 만 남김
    cleaned = re.sub(r"[^A-Za-z0-9_\-:.]", "", text)

    if cleaned:
        return cleaned

    # 전부 제거되면 해시로 대체
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:12]


def build_ids(docs: List[Document]) -> List[str]:
    ids = []

    for i, doc in enumerate(docs, start=1):
        doc_id = ascii_safe(doc.metadata.get("doc_id", "doc"))
        article = ascii_safe(doc.metadata.get("article", ""))
        clause = ascii_safe(doc.metadata.get("clause", ""))

        vector_id = f"{doc_id}::{article}::{clause}::{i:05d}"
        ids.append(vector_id)

    return ids

# def build_ids(docs: List[Document]) -> List[str]:
#     ids = []
#     for i, doc in enumerate(docs, start=1):
#         article = doc.metadata.get("article", "").replace(" ", "")
#         clause = doc.metadata.get("clause", "").replace(" ", "")
#         ids.append(f"{doc.metadata['doc_id']}::{article}::{clause}::{i:05d}")
#     return ids

# -----------------------------
# 5) 기존 데이터 삭제
#    doc_id 기준 삭제
# -----------------------------
def delete_existing_doc(doc_id: str):
    print(f"기존 데이터 삭제 중... doc_id={doc_id}")
    index.delete(
        namespace=NAMESPACE,
        filter={"doc_id": {"$eq": doc_id}},
    )
    print("기존 데이터 삭제 완료")

# -----------------------------
# 6) 새 문서 업로드
# -----------------------------
def reindex_law_docx():
    # 1. 문서 텍스트 추출
    full_text = load_docx_text(DOCX_PATH)

    # 2. 기존 데이터 삭제
    # delete_existing_doc(DOC_ID)

    # 3. 조문 재파싱
    docs = parse_law_to_documents(
        full_text=full_text,
        regulation_name=REGULATION_NAME,
        source=os.path.basename(DOCX_PATH),
        doc_id=DOC_ID,
    )

    # 4. ID 생성
    ids = build_ids(docs)

    # 5. 재저장
    vectorstore.add_documents(documents=docs, ids=ids)

    print(f"재저장 완료: {len(docs)}건")
    print("샘플 metadata:")
    for d in docs[:3]:
        print(d.metadata)

    return docs, ids

# -----------------------------
# 7) 실행
# -----------------------------
docs, ids = reindex_law_docx()

재저장 완료: 274건
샘플 metadata:
{'doc_id': '방송통신설비기술기준', 'regulation_name': '방송통신설비의 기술기준에 관한 규정', 'article': '제1조', 'clause': '', 'source': '방송통신설비기술기준_20240719.docx', 'text': '제1조(목적) 이 영은 「방송통신발전 기본법」'}
{'doc_id': '방송통신설비기술기준', 'regulation_name': '방송통신설비의 기술기준에 관한 규정', 'article': '제28조', 'clause': '제1항', 'source': '방송통신설비기술기준_20240719.docx', 'text': '제1항, 「전기통신사업법」'}
{'doc_id': '방송통신설비기술기준', 'regulation_name': '방송통신설비의 기술기준에 관한 규정', 'article': '제35조의2', 'clause': '', 'source': '방송통신설비기술기준_20240719.docx', 'text': '제35조의2ㆍ'}
